# 06 - SHAP Explainability Analysis
## Predicting Human Intestinal Absorption (HIA)
**Student:** Mohammad Karamath Fardeen | ID: 25251265  
**Supervisor:** Kolawole Adebayo | Maynooth University | 2025-2026

Global SHAP feature importance + Local per-molecule explanations + Lipinski/Veber validation

In [ ]:
import pandas as pd
import numpy as np
import shap
import joblib
import matplotlib.pyplot as plt
import os

FEATURE_COLS = ['MolWt','LogP','TPSA','HBD','HBA','RotBonds',
                'RingCount','AromaticRings','HeavyAtoms',
                'FractionCSP3','MolMR','NumHeteroatoms']
FEATURE_LABELS = {
    'MolWt':'Molecular Weight','LogP':'LogP (Lipophilicity)',
    'TPSA':'TPSA (Polar Surface Area)','HBD':'H-Bond Donors',
    'HBA':'H-Bond Acceptors','RotBonds':'Rotatable Bonds',
    'RingCount':'Ring Count','AromaticRings':'Aromatic Rings',
    'HeavyAtoms':'Heavy Atom Count','FractionCSP3':'Fraction CSP3',
    'MolMR':'Molar Refractivity','NumHeteroatoms':'Heteroatom Count'
}
os.makedirs('results/shap', exist_ok=True)
os.makedirs('results/figures', exist_ok=True)
print('Setup complete!')

## 1. Load Model and Test Data

In [ ]:
model  = joblib.load('results/metrics/xgboost.joblib')
scaler = joblib.load('results/metrics/scaler.joblib')

test_df   = pd.read_csv('data/processed/hia_test_features.csv').dropna(subset=FEATURE_COLS)
X_test    = test_df[FEATURE_COLS].values
X_test_sc = scaler.transform(X_test)
y_test    = test_df['Y'].values
y_pred    = model.predict(X_test_sc)
y_proba   = model.predict_proba(X_test_sc)[:, 1]

print(f'Test samples: {len(y_test)}')

## 2. Global SHAP Analysis

In [ ]:
explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test_sc)
sv = shap_values[1] if isinstance(shap_values, list) else shap_values

# Save SHAP values
shap_df = pd.DataFrame(sv, columns=FEATURE_COLS)
shap_df['true_label'] = y_test
shap_df.to_csv('results/shap/xgboost_shap_values.csv', index=False)

# Print ranking
mean_shap = np.abs(sv).mean(axis=0)
importance = sorted(zip(FEATURE_COLS, mean_shap), key=lambda x: x[1], reverse=True)
print('Global SHAP Feature Importance (Mean |SHAP|):')
print('-' * 45)
for feat, val in importance:
    print(f'  {FEATURE_LABELS[feat]:<30} {val:.4f}')

## 3. SHAP Summary Plot

In [ ]:
plt.figure(figsize=(10, 7))
shap.summary_plot(sv, X_test_sc,
                  feature_names=[FEATURE_LABELS[f] for f in FEATURE_COLS],
                  show=False)
plt.title('SHAP Summary Plot — XGBoost (HIA Test Set)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('results/figures/shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. SHAP Bar Plot

In [ ]:
plt.figure(figsize=(9, 6))
shap.summary_plot(sv, X_test_sc,
                  feature_names=[FEATURE_LABELS[f] for f in FEATURE_COLS],
                  plot_type='bar', show=False)
plt.title('SHAP Feature Importance (Mean |SHAP|) — XGBoost', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('results/figures/shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Local SHAP Explanations (3 Representative Molecules)

In [ ]:
correct = (y_test == y_pred)
base_val = explainer.expected_value[1] if isinstance(explainer.expected_value, (list, np.ndarray)) else explainer.expected_value

# Select 3 representative molecules
cases = {
    'High_Absorption_Correct': np.where(correct & (y_test==1))[0][np.argmax(y_proba[correct & (y_test==1)])],
    'Low_Absorption_Correct':  np.where(correct & (y_test==0))[0][np.argmin(y_proba[correct & (y_test==0)])],
    'Misclassified':           np.where(~correct)[0][0],
}

for label, idx in cases.items():
    explanation = shap.Explanation(
        values=sv[idx], base_values=base_val,
        data=X_test_sc[idx],
        feature_names=[FEATURE_LABELS[f] for f in FEATURE_COLS]
    )
    plt.figure(figsize=(10, 5))
    shap.waterfall_plot(explanation, show=False)
    plt.title(f'Local SHAP — {label} (True: {y_test[idx]}, Pred: {y_pred[idx]}, P={y_proba[idx]:.3f})')
    plt.tight_layout()
    plt.savefig(f'results/figures/shap_local_{label}.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'{label}: True={y_test[idx]}, Pred={y_pred[idx]}, P(High)={y_proba[idx]:.3f}')

## 6. Lipinski's Rule of Five + Veber's Rules Validation

In [ ]:
THRESHOLDS = {
    'MolWt':    {'rule': "Lipinski",  'limit': 500,  'desc': 'Molecular Weight <= 500 Da'},
    'LogP':     {'rule': "Lipinski",  'limit': 5,    'desc': 'LogP <= 5'},
    'HBD':      {'rule': "Lipinski",  'limit': 5,    'desc': 'H-Bond Donors <= 5'},
    'HBA':      {'rule': "Lipinski",  'limit': 10,   'desc': 'H-Bond Acceptors <= 10'},
    'TPSA':     {'rule': "Veber",     'limit': 140,  'desc': 'TPSA <= 140 A²'},
    'RotBonds': {'rule': "Veber",     'limit': 10,   'desc': 'Rotatable Bonds <= 10'},
}

results = []
print('Lipinski/Veber Rule Validation')
print('=' * 65)
for feat, info in THRESHOLDS.items():
    high_df = test_df[test_df['Y'] == 1]
    low_df  = test_df[test_df['Y'] == 0]
    high_viol = (high_df[feat] > info['limit']).mean() * 100
    low_viol  = (low_df[feat]  > info['limit']).mean() * 100
    confirmed = '✓ CONFIRMED' if low_viol > high_viol else '⚠ Not confirmed'
    print(f"{info['desc']} ({info['rule']})")
    print(f"  High absorption: {high_df[feat].mean():.2f} mean | {high_viol:.1f}% violating")
    print(f"  Low absorption:  {low_df[feat].mean():.2f} mean | {low_viol:.1f}% violating")
    print(f"  {confirmed}\n")
    results.append({'Feature': feat, 'Rule': info['rule'],
                    '% High Violating': round(high_viol,1),
                    '% Low Violating': round(low_viol,1),
                    'Confirmed': confirmed})

pd.DataFrame(results).to_csv('results/shap/lipinski_validation.csv', index=False)
print('Saved to results/shap/lipinski_validation.csv')